# Electricity Load Forecasting (h+1) — XGBoost Machine Learning Model

The goal of this notebook is to build a **XGBoost** (Extreme Gradient Boosting) model to predict **hourly electricity demand one hour ahead (h+1)** using historical load, weather, and calendar features.

This notebook follows **time-series best practices**:
- No data leakage
- Time-aware train/validation/test splits
- Baseline comparison
- Progressive model complexity

## 1. Problem formulation

We formulate the task as a **supervised regression problem** with:
* **Features (X)** available at time t:
    * Load at time t
    * Load lags: t-1, t-24, t-168
    * Temperature at time t
    * Calendar features (hour, weekday, week of year)
* **Target (y)**:
    * Load at time t+1
    
Mathematically:
$$\hat{y}_{t+1} = f(X_t)$$

## 2. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np

import xgboost as xgb

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error

## 3. Project Path and Parameters

In [ ]:
FEATURED_BASE_PATH = "../data/featured"

In [ ]:
# Parameters
country = "FR"
years = list(range(2015, 2024+1))

## 4. Load feature-engineered dataset

The dataset consists of **hourly electricity demand and weather data** for France between 2015 and 2024.

Key characteristics:
* Hourly resolution
* Strong daily, weekly and annual seasonality
* High temporal autocorrelation
* Strong dependency on temperature

In [ ]:
# Load feature-engineered data
dfs = []
for year in years:
    path = f"{FEATURED_BASE_PATH}/country={country}/year={year}/load_forecasting_features.parquet"
    dfs.append(pd.read_parquet(path))

df = pd.concat(dfs).sort_index()

In [ ]:
df.info()

In [ ]:
df.head()

## 5. Feature matrix and target

In [ ]:
# Transform datetime to index
df = df.assign(datetime=df["datetime"]).set_index("datetime").sort_index()

In [ ]:
# Check that the index is sorted
assert df.index.is_monotonic_increasing

In [ ]:
# Target column
target_col = "target_load_t+1"

# Features columns
feature_cols = [
    "load_t",
    "load_t-1",
    "load_t-24",
    "load_t-168",
    "temperature_t", # For the moment, we only use temperature from weather data
    "hour",
    "is_weekday",
    "week_of_year",
]

X = df[feature_cols]
y = df[target_col]

## 5. Time-based data split

In time series forecasting, **random splits are invalid** because future information would leak into the past.

We use a chronological split:
* Train: 2015–2022
* Validation: 2023
* Test: 2024

In [ ]:
# Split dates : train (2015-2022), val (2023), test (2024)
train_end = pd.Timestamp("2022-12-31 23:00:00", tz="UTC")
val_end = pd.Timestamp("2023-12-31 23:00:00", tz="UTC")

X_train = X.loc[:train_end]
y_train = y.loc[:train_end]

X_val = X.loc[train_end:val_end]
y_val = y.loc[train_end:val_end]

X_test = X.loc[val_end:]
y_test = y.loc[val_end:]

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

## 6. XGBoost Model